# Sparse Auto-Encoder

## Importing the libraries

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

## Importing the dataset

In [2]:
movies = pd.read_csv('ml-1m/movies.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')
users = pd.read_csv('ml-1m/users.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')
ratings = pd.read_csv('ml-1m/ratings.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')

## Preparing the training set and the test set

In [3]:
training_set = pd.read_csv('ml-100k/u1.base', delimiter = '\t')
training_set = np.array(training_set, dtype = 'int')

test_set = pd.read_csv('ml-100k/u1.test', delimiter = '\t')
test_set = np.array(test_set, dtype = 'int')

## Getting the number of users and movies

In [4]:
nb_users = int(max(max(training_set[:, 0]), max(test_set[:, 0])))
nb_movies = int(max(max(training_set[:, 1]), max(test_set[:, 1])))

## Converting the data into an array with users in lines and movies in columns

In [5]:
def convert(data):
    new_data = []

    for id_users in range(1, nb_users + 1):

        id_movies = data[:, 1][data[:, 0] == id_users]
        id_ratings = data[:, 2][data[:, 0] == id_users]

        ratings = np.zeros(nb_movies)
        ratings[id_movies - 1] = id_ratings

        new_data.append(list(ratings))

    return new_data

In [6]:
training_set = convert(training_set)
test_set = convert(test_set)

## Converting the data into Torch tensors

In [7]:
training_set = torch.FloatTensor(training_set)
test_set = torch.FloatTensor(test_set)

## Creating the architecture of the Neural Network

In [8]:
class SAE(nn.Module):

  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(nb_movies, 20)
    self.fc2 = nn.Linear(20, 10)
    self.fc3 = nn.Linear(10, 20)
    self.fc4 = nn.Linear(20, nb_movies)
    self.activation = nn.Sigmoid()

  def forward(self, x):
    x = self.activation(self.fc1(x))
    x = self.activation(self.fc2(x))
    x = self.activation(self.fc3(x))
    x = self.fc4(x)
    return x

In [9]:
sae = SAE()
criterion = nn.MSELoss()
optimizer = torch.optim.RMSprop(
    sae.parameters(),
    lr = 0.01,
    weight_decay = 0.5
)

## Training the SAE

In [10]:
epochs = 200

In [11]:
for epoch in range(epochs):
  train_loss = 0
  counter = 0

  for id_user in range(nb_users):
      input = training_set[id_user].unsqueeze(0)
      target = input.clone()

      if torch.sum(target.data > 0) > 0:
          output = sae(input)
          target.require_grad = False
          output[target == 0] = 0

          loss = criterion(output, target)
          mean_corrector = nb_movies / float(torch.sum(target.data > 0) + 1e-10)
          loss.backward()

          train_loss += np.sqrt(loss.item() * mean_corrector)
          counter += 1

          optimizer.step()

  epoch_loss = train_loss / counter
  print(f"Epoch: {epoch + 1}/{epochs} - Loss: {epoch_loss:.4f}")

Epoch: 1/200 - Loss: 1.7712
Epoch: 2/200 - Loss: 1.0966
Epoch: 3/200 - Loss: 1.0535
Epoch: 4/200 - Loss: 1.0384
Epoch: 5/200 - Loss: 1.0307
Epoch: 6/200 - Loss: 1.0266
Epoch: 7/200 - Loss: 1.0238
Epoch: 8/200 - Loss: 1.0219
Epoch: 9/200 - Loss: 1.0208
Epoch: 10/200 - Loss: 1.0197
Epoch: 11/200 - Loss: 1.0187
Epoch: 12/200 - Loss: 1.0184
Epoch: 13/200 - Loss: 1.0178
Epoch: 14/200 - Loss: 1.0175
Epoch: 15/200 - Loss: 1.0171
Epoch: 16/200 - Loss: 1.0168
Epoch: 17/200 - Loss: 1.0167
Epoch: 18/200 - Loss: 1.0164
Epoch: 19/200 - Loss: 1.0164
Epoch: 20/200 - Loss: 1.0161
Epoch: 21/200 - Loss: 1.0161
Epoch: 22/200 - Loss: 1.0157
Epoch: 23/200 - Loss: 1.0159
Epoch: 24/200 - Loss: 1.0156
Epoch: 25/200 - Loss: 1.0157
Epoch: 26/200 - Loss: 1.0156
Epoch: 27/200 - Loss: 1.0154
Epoch: 28/200 - Loss: 1.0149
Epoch: 29/200 - Loss: 1.0127
Epoch: 30/200 - Loss: 1.0111
Epoch: 31/200 - Loss: 1.0103
Epoch: 32/200 - Loss: 1.0086
Epoch: 33/200 - Loss: 1.0080
Epoch: 34/200 - Loss: 1.0044
Epoch: 35/200 - Loss: 1

## Testing the SAE

In [12]:
test_loss = 0
counter = 0

for id_user in range(nb_users):
    input = training_set[id_user].unsqueeze(0)
    target = test_set[id_user].unsqueeze(0)

    if torch.sum(target.data > 0) > 0:
        output = sae(input)
        target.require_grad = False
        output[target == 0] = 0

        loss = criterion(output, target)
        mean_corrector = nb_movies / float(torch.sum(target.data > 0) + 1e-10)

        test_loss += np.sqrt(loss.item() * mean_corrector)
        counter += 1

print(f"Test Loss: {epoch_loss:.6f}")

Test Loss: 0.914813


## Predicting for one user

In [32]:
user_id = 5
user = training_set[user_id]

with torch.no_grad():
    prediction = sae(user)

print(prediction)

tensor([3.4794, 2.7813, 2.5198,  ..., 1.8262, 2.7858, 2.5575])


## Recommending Movies

In [33]:
user_id = 5
user = training_set[user_id]

with torch.no_grad():
    prediction = sae(user)

prediction = prediction.detach().numpy()
unseen = training_set[user_id].numpy() == 0
scores = prediction.copy()
scores[~unseen] = -1

recommended = np.argsort(scores)[::-1][:10]
print(recommended)

[1499 1466 1598  317  356 1366 1641 1652  514 1448]


## Printing the name of movies

In [34]:
for movie in recommended:
    print(f"{movie:<5}: {movies.iloc[movie, 1]}")

1499 : Shall We Dance? (Shall We Dansu?) (1996)
1466 : Inventing the Abbotts (1997)
1598 : I Know What You Did Last Summer (1997)
317  : Suture (1993)
356  : I Love Trouble (1994)
1366 : Jaws (1975)
1641 : Jackal, The (1997)
1652 : Butcher Boy, The (1998)
514  : Road to Wellville, The (1994)
1448 : Private Parts (1997)
